# QDPX to Belege Converter

Converts MAXQDA QDPX project files into the `belege_*.xlsx` format used by `belege.ipynb`.

**Mapping:**
- Each unique coded passage (same document + position) → one belege row
- Multiple codes on the same passage → merged into comma-separated `themen`
- Source document name → `quelle` column
- Quote text → `zitat` column
- Character position → `timecode` column (as reference)


In [ ]:
from pathlib import Path
from collections import defaultdict
import zipfile
import xml.etree.ElementTree as ET

from qdpxlib import QDPXFile
from openpyxl import Workbook


## Configuration

In [ ]:
# Input QDPX file
qdpx_path = Path("../data/maxqda/prin_22_stern_1_2.qdpx")

# Output directory (will create belege xlsx here)
output_dir = Path("../data/schede mappatura/stern_IS_S_00142")

# Person identifier for betrifft_personen and quelle_sprecher
person_identifier = "IS_S_00142"

# Prefix for beleg_id generation
beleg_prefix = "bel_josef_qdpx"


## Extract source document metadata from QDPX

In [ ]:
def extract_source_names(qdpx_path):
    """Extract source document GUID → name mapping from the QDPX project XML."""
    source_map = {}
    with zipfile.ZipFile(str(qdpx_path), 'r') as zf:
        content = zf.read('project.qde')
        root = ET.fromstring(content)
        ns = root.tag.split('}')[0] + '}' if '}' in root.tag else ''
        for elem in root.iter(f'{ns}TextSource'):
            guid = elem.attrib.get('guid', '')
            name = elem.attrib.get('name', '')
            if guid and name:
                source_map[guid] = name
    return source_map

source_names = extract_source_names(qdpx_path)
print(f"Source documents: {len(source_names)}")
for guid, name in source_names.items():
    print(f"  {name} ({guid})")


## Parse QDPX and group codings by passage

In [ ]:
from collections import Counter
qdpx = QDPXFile(str(qdpx_path))

print(f"Total codes: {len(qdpx.codes)}")
print(f"Total codings: {len(qdpx.codings)}")

# Group codings by (doc_guid, start_pos, end_pos) to merge tags
passages = defaultdict(lambda: {"quote": "", "tags": set(), "doc_guid": ""})

for coding in qdpx.codings:
    doc_guid = coding[0]
    quote = coding[1]
    pos = coding[3]  # (start, end) tuple
    tags = coding[4]  # list of code names

    key = (doc_guid, pos)
    passages[key]["quote"] = quote
    passages[key]["doc_guid"] = doc_guid
    passages[key]["tags"].update(tags)

print(f"Unique passages: {len(passages)}")

# Sort by document and position for consistent ordering
sorted_keys = sorted(passages.keys(), key=lambda k: (k[0], k[1][0]))
print("\nPassages per source:")
doc_counts = Counter(k[0] for k in sorted_keys)
for guid, count in doc_counts.items():
    print(f"  {source_names.get(guid, guid)}: {count}")


## Preview passages

In [ ]:
for i, key in enumerate(sorted_keys[:5]):
    p = passages[key]
    doc_name = source_names.get(p["doc_guid"], p["doc_guid"])
    pos = key[1]
    print(f"--- Passage {i+1} ---")
    print(f"  Source: {doc_name}")
    print(f"  Position: {pos[0]}–{pos[1]}")
    print(f"  Tags: {', '.join(sorted(p['tags']))}")
    print(f"  Quote: {p['quote'][:100]}...")
    print()


## Convert to belege xlsx format

Creates an xlsx file with the standard belege columns:
`beleg_id, quelle, interview_id, interview_datum, quelle_sprecher, betrifft_personen, timecode, themen, zitat, markierung, event_ids, event_ids_confidence, notizen`


In [ ]:
def qdpx_to_belege_xlsx(passages, sorted_keys, source_names,
                       output_path, person_identifier, beleg_prefix):
    """Convert grouped QDPX passages to belege xlsx format."""
    wb = Workbook()
    ws = wb.active
    ws.title = "Belege"

    # Row 1: Title
    ws.append(["Belege (from QDPX)", None, None, None, None, None,
               f"{len(sorted_keys)} Belege from {len(source_names)} sources"])

    # Row 2: Blank
    ws.append([None] * 13)

    # Row 3: Headers
    headers = [
        "beleg_id", "quelle", "interview_id", "interview_datum",
        "quelle_sprecher", "betrifft_personen", "timecode", "themen",
        "zitat", "markierung", "event_ids", "event_ids_confidence", "notizen"
    ]
    ws.append(headers)

    # Data rows
    for i, key in enumerate(sorted_keys):
        p = passages[key]
        doc_name = source_names.get(p["doc_guid"], p["doc_guid"])
        pos = key[1]

        beleg_id = f"{beleg_prefix}_{i+1:04d}"
        quelle = doc_name
        themen = ", ".join(sorted(p["tags"]))
        quote = p["quote"].strip()
        timecode = f"pos {pos[0]}–{pos[1]}"

        row = [
            beleg_id,           # beleg_id
            quelle,             # quelle
            None,               # interview_id
            None,               # interview_datum
            person_identifier,  # quelle_sprecher
            person_identifier,  # betrifft_personen
            timecode,           # timecode
            themen,             # themen
            quote,              # zitat
            None,               # markierung
            None,               # event_ids
            None,               # event_ids_confidence
            None,               # notizen
        ]
        ws.append(row)

    # Auto-adjust column widths
    for col in ws.columns:
        max_len = 0
        for cell in col:
            if cell.value:
                max_len = max(max_len, min(len(str(cell.value)), 50))
        ws.column_dimensions[col[0].column_letter].width = max_len + 2

    wb.save(output_path)
    return len(sorted_keys)

# Generate output filename from QDPX filename
qdpx_stem = qdpx_path.stem  # e.g. "prin_22_stern_1_2"
output_path = output_dir / f"belege_{qdpx_stem}.xlsx"

count = qdpx_to_belege_xlsx(
    passages, sorted_keys, source_names,
    output_path, person_identifier, beleg_prefix
)
print(f"Wrote {count} passages to {output_path}")


## Verify output

In [ ]:
import openpyxl

wb = openpyxl.load_workbook(output_path)
ws = wb["Belege"]
rows = list(ws.iter_rows(values_only=True))

print(f"Title: {rows[0][0]}")
print(f"Headers: {rows[2]}")
print(f"Data rows: {len(rows) - 3}")
print("\nFirst 3 rows:")
for row in rows[3:6]:
    cells_list = list(row)
    print(f"  ID: {cells_list[0]}")
    print(f"    Source: {cells_list[1]}")
    print(f"    Timecode: {cells_list[6]}")
    print(f"    Themen: {cells_list[7]}")
    print(f"    Quote: {str(cells_list[8])[:80]}...")
    print()


## Export MAXQDA coding system as Mermaid taxonomy

Reads the hierarchical code structure from the QDPX project XML and exports it as a Mermaid mindmap diagram.

In [ ]:
import re as _re

def _mermaid_id(name):
    """Sanitize a code name into a valid Mermaid node ID."""
    sanitized = _re.sub(r'[^A-Za-z0-9_]', '_', name)
    # Ensure it doesn't start with a digit
    if sanitized and sanitized[0].isdigit():
        sanitized = '_' + sanitized
    return sanitized

def _build_code_tree(qdpx_path):
    """Parse the QDPX XML and return the top-level Codes element."""
    with zipfile.ZipFile(str(qdpx_path), 'r') as zf:
        content = zf.read('project.qde')
        # Clean non-ASCII (same as qdpxlib does)
        content = bytes(b for b in content if b < 128)
        root = ET.fromstring(content)

    # Walk to CodeBook > Codes
    for codebook in root.iter():
        tag = codebook.tag.split('}')[-1] if '}' in codebook.tag else codebook.tag
        if tag == 'CodeBook':
            for codes in codebook:
                ct = codes.tag.split('}')[-1] if '}' in codes.tag else codes.tag
                if ct == 'Codes':
                    return codes
    return None

def _code_children(elem):
    """Yield direct child Code elements."""
    for child in elem:
        tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
        if tag == 'Code':
            yield child

def _render_mermaid_node(elem, indent):
    """Recursively render a Code element as Mermaid mindmap lines."""
    lines = []
    for code in _code_children(elem):
        name = code.get('name', '?')
        mid = _mermaid_id(name)
        # Use quoted label if it differs from the ID
        if mid != name:
            lines.append(f"{'  ' * indent}{mid}[\"{name}\"]")
        else:
            lines.append(f"{'  ' * indent}{mid}")
        lines.extend(_render_mermaid_node(code, indent + 1))
    return lines

codes_root = _build_code_tree(qdpx_path)
if codes_root is None:
    raise ValueError("CodeBook/Codes not found in QDPX")

mermaid_lines = ["mindmap", "  root((MAXQDA Coding System))"]
mermaid_lines.extend(_render_mermaid_node(codes_root, 2))

output_mmd = Path("../docs/mqda_coding_taxonomy.mmd")
output_mmd.write_text("\n".join(mermaid_lines) + "\n", encoding="utf-8")
print(f"Wrote {len(mermaid_lines)} lines to {output_mmd}")
print("\nPreview (first 30 lines):")
for line in mermaid_lines[:30]:
    print(line)